# S09 · Three classifiers, one job — Logistic Regression, KNN and SVM

A bank is deciding a loan: will this borrower repay, or default? A payments
company watching a transaction: genuine, or fraud? These are **classification**
problems — predict a *category*, not a number. Last session we drew a line
*through* points to predict a price. Today we draw a line *between* groups, and
we have three classic ways to draw it: **logistic regression**, **KNN** and
**SVM**. We fit all three on the same real data and judge them with the same
ruler.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to classification? This session is one idea repeated three times: a model
  that separates two groups. The idea carries from Session 7 — you still call
  `.fit(...)` then `.predict(...)`; only the output changes from a number to a
  category.
- Already confident with code or with these classifiers? Skip ahead to the cells
  marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib and scikit-learn.
# Google Colab already ships all three, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                                   # fast maths on lists of numbers
import matplotlib.pyplot as plt                       # drawing charts
from sklearn.datasets import load_iris, make_moons    # real iris data + a curvy toy set
from sklearn.model_selection import train_test_split  # split train vs test
from sklearn.linear_model import LogisticRegression   # classifier 1
from sklearn.neighbors import KNeighborsClassifier    # classifier 2
from sklearn.svm import SVC                            # classifier 3
from sklearn.preprocessing import StandardScaler       # put features on one scale
from sklearn.pipeline import make_pipeline             # glue scaler + model together

## Step 1 — a real classification problem: iris flowers

We use a real, classic dataset that ships inside scikit-learn: 150 iris flowers,
each measured in four ways (sepal length, sepal width, petal length, petal
width) and each belonging to one of three species. Predicting the species from
the measurements is exactly the same task as predicting "default" from a
borrower's numbers — just with three categories instead of two.

In [ ]:
# Load the iris dataset (built into scikit-learn, so no download needed).
iris = load_iris()
measurements = iris.data     # the four numbers measured per flower
target = iris.target          # the species: 0, 1 or 2

print("measurements shape:", measurements.shape, "  (150 flowers, 4 measurements each)")
print("species          :", iris.target_names)
print("flowers per species:", [int(np.sum(target == i)) for i in range(3)])

## Step 2 — split into a training part and a test part

Same honest habit as Session 7: train on one part of the data and judge on a
part the models have never seen. We keep 30% aside as the test set.

In [ ]:
# 70% to train on, 30% kept aside for an honest test.
train_x, test_x, train_label, test_label = train_test_split(
    measurements, target, test_size=0.3, random_state=42)

print("training flowers:", train_x.shape[0])
print("test flowers    :", test_x.shape[0])

## Step 3 — classifier 1: logistic regression (a straight line, plus a probability)

Logistic regression = linear regression + sigmoid. It first computes the same
weighted sum you met in Session 7, then passes it through the **sigmoid** curve
that squashes any number into a probability between 0 and 1. It predicts the
species with the highest probability — and, like in the last session, it reports
*how sure* it is. For three species it learns one such rule per species.

In [ ]:
# Fit the classifier with the same .fit(...) habit as Session 7.
logistic = LogisticRegression(max_iter=1000)
logistic.fit(train_x, train_label)

print("Logistic regression accuracy on the test set:",
      round(logistic.score(test_x, test_label), 3))
print()
print("Predicted probabilities for the first 3 test flowers:")
probabilities = logistic.predict_proba(test_x[:3])
print("   " + "  ".join(f"P({name})" for name in iris.target_names))
print(probabilities.round(3))

## Step 4 — classifier 2: KNN (ask your neighbours)

KNN (k-nearest neighbours) is a completely different idea: it does **no
training at all**. It simply stores the training flowers. To classify a new
flower it measures the **Euclidean** (straight-line) distance to every stored
flower, finds the `k` nearest ones, and they vote. We use `k = 3`, the classic
choice.

In [ ]:
# k = 3: the 3 nearest stored flowers vote on each new flower.
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(train_x, train_label)

print("KNN (k = 3) accuracy on the test set:", round(knn.score(test_x, test_label), 3))

## Step 5 — classifier 3: SVM (the widest street)

The support vector machine (**SVM**) draws the boundary that leaves the widest
empty gap — the "street" — between the two groups. The flowers sitting right on
the street's edge, the **support vectors**, are the only ones that matter: move
any other flower and the boundary stays put. The **RBF kernel** lets that
boundary bend into a smooth curve, which helps when the groups are not split by
a straight line.

In [ ]:
# RBF kernel: a smooth, curving boundary. Default C balances street width vs mistakes.
svm = SVC(kernel="rbf")
svm.fit(train_x, train_label)

print("SVM (RBF) accuracy on the test set:", round(svm.score(test_x, test_label), 3))

## Step 6 — judge all three with the SAME metrics

Here is the whole message of this session: the classifier changed, the way we
judge it does not. We score all three with **accuracy**, **F1** and the
**confusion matrix** — the same metrics you will use for any classifier from now
on.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

models = {"logistic regression": logistic, "KNN (k=3)": knn, "SVM (RBF)": svm}

print(f"{'classifier':<20}{'accuracy':>9}{'F1':>7}")
for name, model in models.items():
    prediction = model.predict(test_x)
    acc = accuracy_score(test_label, prediction)
    f1 = f1_score(test_label, prediction, average="macro")
    print(f"{name:<20}{acc:>9.3f}{f1:>7.3f}")

print("\nConfusion matrix for each model (rows = true species, cols = predicted):")
for name, model in models.items():
    print(f"\n{name}:")
    print(confusion_matrix(test_label, model.predict(test_x)))

## Step 7 — the same three, on data a straight line cannot split

On iris all three do well — the species are nicely separable. Real data is rarely
that tidy. So let us try the same three classifiers on `make_moons`: two
interleaving crescents that **no straight line can separate**. Watch the
straight line struggle while the two curved boundaries keep up. KNN and SVM both
measure distances, so we scale the features first with a `StandardScaler` inside
a Pipeline (learned on the training part only — no peeking at the test set).

In [ ]:
# Two interleaving moons, 300 points, with some noise. Same data every run.
X, y = make_moons(n_samples=300, noise=0.25, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

three_models = {
    "logistic regression": LogisticRegression(),
    "KNN (k=15, scaled)":  make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "SVM (RBF, scaled)":   make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}

print(f"{'classifier':<24}{'accuracy':>9}{'F1':>7}")
for name, model in three_models.items():
    prediction = model.fit(X_train, y_train).predict(X_test)
    print(f"{name:<24}{accuracy_score(y_test, prediction):>9.3f}"
          f"{f1_score(y_test, prediction):>7.3f}")

print("\nThe straight line can't keep up. Curved boundaries follow the moons.")

## Step 8 — the boundaries, side by side

Numbers are honest, but a picture makes the difference obvious. We colour the
plane by what each model predicts and drop the test points on top: a straight
line where the two curved boundaries hug the moons.

In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, 300),
                             np.linspace(y_min, y_max, 300))
grid = np.c_[grid_x.ravel(), grid_y.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, model) in zip(axes, three_models.items()):
    zone = model.predict(grid).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, zone, alpha=0.15,
                levels=[-0.5, 0.5, 1.5], colors=["#2E75B6", "#C0392B"])
    ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1],
               color="#2E75B6", s=18, edgecolor="white", linewidth=0.4)
    ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1],
               color="#C0392B", s=18, marker="^", edgecolor="white", linewidth=0.4)
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("The same data, three different boundaries")
plt.show()

## What you just did

You met **three classifiers** — logistic regression (a straight line plus a
probability), **KNN** (ask the nearest stored cases to vote) and **SVM** (leave
the widest street between the groups) — and judged all three with the same
accuracy, F1 and confusion matrix. On easy, separable data all three shine; on
curvy data the straight line loses. **No single classifier is best** — the shape
of the data decides, and metrics, not opinions, make the call.

The Stretch cells below are optional extras for the confident.

### Stretch (optional) — the sigmoid and log-loss, by hand

Skip this if you are new to code. Logistic regression = linear + sigmoid. The
**sigmoid** squashes any score into (0, 1), and **cross-entropy** (log-loss) is
the loss it trains on: a confident *wrong* answer costs a lot, a confident
*right* answer almost nothing. We take a two-species slice of iris, fit logistic
regression, rebuild the probability by hand, and check our log-loss matches
scikit-learn.

In [ ]:
# Keep only two species so we have a clean binary problem: setosa (0) vs versicolor (1).
keep = target < 2
bin_x, bin_y = measurements[keep], target[keep]

bin_train_x, bin_test_x, bin_train_label, bin_test_label = train_test_split(
    bin_x, bin_y, test_size=0.3, random_state=42)

binary = LogisticRegression(max_iter=1000)
binary.fit(bin_train_x, bin_train_label)
prob_of_one = binary.predict_proba(bin_test_x)[:, 1]   # P(versicolor)

# Rebuild the probability for the first test flower by hand.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z_by_hand = np.dot(bin_test_x[0], binary.coef_[0]) + binary.intercept_[0]
prob_by_hand = sigmoid(z_by_hand)

print("score z by hand        :", round(z_by_hand, 4))
print("sigmoid(z) by hand     :", round(prob_by_hand, 4))
print("predict_proba (sklearn):", round(prob_of_one[0], 4))

# Cross-entropy / log-loss by hand, then compare with scikit-learn.
from sklearn.metrics import log_loss
prob_of_truth = np.where(bin_test_label == 1, prob_of_one, 1 - prob_of_one)
loss_by_hand = np.mean(-np.log(prob_of_truth))

print()
print("log-loss by hand   :", round(loss_by_hand, 4))
print("log-loss (sklearn) :", round(log_loss(bin_test_label, prob_of_one), 4))

### Stretch (optional) — turn the knobs

Skip this unless you want to experiment. Two knobs to play with on the curvy
moons:

- **KNN's `k`**: a small `k` hugs the data (a jumpy boundary); a large `k`
  smooths it out — until it underfits. Try 1 and 60 and compare the scores.
- **SVM's kernel and `C`**: `kernel="linear"` forces a straight boundary again;
  `C` trades a wider margin against fewer training mistakes.

In [ ]:
for k in [1, 15, 60]:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    m.fit(X_train, y_train)
    print(f"KNN  k={k:<3d}  F1 = {f1_score(y_test, m.predict(X_test)):.3f}")

print()

for kernel, C in [("linear", 1.0), ("rbf", 0.5), ("rbf", 10.0)]:
    m = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=C))
    m.fit(X_train, y_train)
    print(f"SVM  kernel={kernel:<6}  C={C:<4}  F1 = {f1_score(y_test, m.predict(X_test)):.3f}")

### Stretch (optional) — why scaling matters for KNN and SVM

Skip this unless you want to see the trap. KNN and SVM both measure distances,
and a distance adds up contributions from every feature. If one feature has a
much larger range, it silently dominates. Let us build a two-feature problem,
inflate one feature 1,000×, and watch KNN and SVM fail until we scale.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=300, n_features=2, n_informative=2,
                           n_redundant=0, random_state=42)
X_lopsided = X.copy()
X_lopsided[:, 1] *= 1000          # inflate the second feature 1000x

for name, model, scale in [
    ("KNN raw",     KNeighborsClassifier(15), False),
    ("KNN scaled",  KNeighborsClassifier(15), True),
    ("SVM raw",     SVC(kernel="rbf"),        False),
    ("SVM scaled",  SVC(kernel="rbf"),        True),
]:
    if scale:
        model = make_pipeline(StandardScaler(), model)
    xx_tr, xx_te, yy_tr, yy_te = train_test_split(X_lopsided, y,
                                                  test_size=0.3, random_state=42)
    model.fit(xx_tr, yy_tr)
    print(f"{name:<12} accuracy {accuracy_score(yy_te, model.predict(xx_te)):.3f}")

print("\nRaw: the big-range feature dominates the distance. Scaled: both count equally.")

## Your turn (5 minutes)

Small changes, so the ideas stick. Copy a line from above into the empty cell
and edit it.

1. Change KNN's `n_neighbors` on iris from 3 to 15. Does iris accuracy drop?
2. On the moons, set the SVM kernel to `"linear"`. What happens to its score, and why?
3. In one sentence in a text cell: if a problem's two groups are neatly
   separated by a straight line, which classifier would you reach for — and why?

In [ ]:
# Your turn — write your code here.